In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
from collections import Counter
from collections import defaultdict
from tqdm import tqdm
from Bio.Align import PairwiseAligner

In [ ]:
data = pd.read_csv('../large_files/8287_28_S28_db-pass.airr.tsv',sep='\t')
data.head()

In [ ]:
def fasta_to_df(file):
    lines = open(file,'r').readlines()
    df = []
    for i in range(0,len(lines),2):
        df.append([lines[i][1:].strip(),lines[i+1].strip()])
    return pd.DataFrame(df,columns=['name','sequence'])

def find_match(seq, expected):
    best_match = ''
    best_muts = np.inf
    mut_locs = np.array([])

    for i,row in expected.iterrows():
        aligner = PairwiseAligner()
        aligner.mode = 'global'
        aligner.open_gap_score = -1
        
        algn = aligner.align(seq,row['sequence'])[0]
        seq_array = np.char.array(list(algn[0,:]))
        exp_array = np.char.array(list(algn[1,:]))

        exp_not_gap = np.where(exp_array!='-')
        start = exp_not_gap[0][0]
        end = exp_not_gap[0][-1]+1

        mut_binary = seq_array[start:end]!=exp_array[start:end]
        n_muts = np.sum(mut_binary)
        if n_muts < best_muts:
            best_muts = n_muts
            best_match = row['name']

            gap_count = np.cumsum(exp_array[start:end]=='-')
            mut_locs = np.where(mut_binary)[0]
            mut_locs = mut_locs - gap_count[mut_locs]

    return best_match, best_muts, mut_locs

In [ ]:
expected = fasta_to_df('expected_seqs.fasta')
expected = expected.loc[[1],:]
expected.head()

In [ ]:
data['match_info'] = data['sequence'].map(lambda s: find_match(s,expected))
data['nmut'] = data['match_info'].map(lambda x: x[1])
data['match_name'] = data['match_info'].map(lambda x: x[0])
data['mut_locs'] = data['match_info'].map(lambda x: x[2])
del data['match_info']

data.head()

In [ ]:
plt.figure(figsize=[13,4.3])

ax = plt.subplot(1,3,1)
mut_counts = defaultdict(int)
for i,row in data.iterrows():
    mut_counts[row['nmut']] += row['duplicate_count']

x = np.arange(140)
plt.bar(x,[mut_counts[i] for i in x],width=1,alpha=0.75)
plt.semilogy()
plt.xlabel('Number of Mutations',fontsize=16)
plt.ylabel('Number of Reads',fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
ax.text(-0.15, 1.05, 'A', fontsize=32, transform=ax.transAxes)

ax = plt.subplot(1,3,2)
sns.histplot(data['duplicate_count'],bins=np.logspace(0,5.5,30))
plt.loglog()
plt.xlabel('Number of Duplicates',fontsize=16)
plt.ylabel('Number of Distinct Sequences',fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
ax.text(-0.15, 1.05, 'B', fontsize=32, transform=ax.transAxes)

ax = plt.subplot(1,3,3)
L = len(expected.loc[1,'sequence'])

mut_counts = np.zeros([L])
for i,row in data.iterrows():
    for idx in row['mut_locs']:
        mut_counts[idx] += row['duplicate_count']

plt.bar(np.arange(0,L,3),mut_counts[::3]+mut_counts[1::3]+mut_counts[2::3],width=3,alpha=0.75)
plt.xlabel('Sequence Position',fontsize=16)
plt.ylabel('Number of Reads',fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
ax.text(-0.15, 1.05, 'C', fontsize=32, transform=ax.transAxes)

plt.subplots_adjust(wspace=0.4)
plt.savefig('figS13.png',dpi=300,bbox_inches='tight')

In [ ]:
(data['nmut']*data['duplicate_count']).sum()/data['duplicate_count'].sum()

In [ ]:
seq_err = (((data['duplicate_count']<=3) & (data['nmut']>0))*data['duplicate_count']).sum()/data['duplicate_count'].sum()
real = ((data['nmut']==0)*data['duplicate_count']).sum()/data['duplicate_count'].sum()
pcr_err = 1 - (seq_err + real)
print(real, pcr_err, seq_err)

In [ ]:
(data['nmut']*data['duplicate_count']).sum()

In [ ]:
np.sum(mut_counts)

In [ ]:
mut_counts

In [ ]:
(((data['duplicate_count']<=10) & (data['nmut']>0))*data['duplicate_count']).sum()/data['duplicate_count'].sum()